<a href="https://colab.research.google.com/github/yaesur/business_python/blob/main/%5B%EB%85%B8%ED%9B%84%EB%8F%84%5D_%EC%BB%A4%ED%8A%B8%EB%9D%BC%EC%9D%B8%EA%B3%BC_%EC%83%81%EA%B4%80%EA%B3%84%EC%88%98_%EB%B6%84%EC%84%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 통합건물대장_표제부 파일

In [12]:
import pandas as pd
import glob
import os
import warnings
warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

# 통합할 파일들이 모여있는 규칙 설정

file_pattern = "*03. 표제부*.csv"
file_list = glob.glob(file_pattern)


print(f"{len(file_list)}")

for f in file_list:
    print(f"  -> {f}")


if len(file_list) == 0:
    print(" error ")
else:
    combined_chunks = []
    total_rows = 0

    print("\n자치구별 데이터 결합 및 대용량 최적화 작업 시작...")

    for file in file_list:
        print(f"   [읽는 중] {file} ...", end="")

        # 메모리 절약을 위해 Chunk 단위로 리딩
        chunk_list = []
        for chunk in pd.read_csv(file, chunksize=30000, low_memory=False):
            # 행정코드 유실 방지를 위해 코드를 string으로 안전하게 변환
            for col in ['시군구코드', '법정동코드', '번', '지']:
                if col in chunk.columns:
                    chunk[col] = chunk[col].astype(str).str.split('.').str[0].str.zfill(4 if col in ['번', '지'] else 5)
            chunk_list.append(chunk)

        df_file = pd.concat(chunk_list, ignore_index=True)
        combined_chunks.append(df_file)

        print(f"{len(df_file):,}")
        total_rows += len(df_file)

    # 하나의 데이터프레임으로 최종 병합
    print("\n모든 구 데이터를 하나의 마스터 파일로 통합 중...")
    df_final_integrated = pd.concat(combined_chunks, ignore_index=True)

    # 최종 결과물 CSV 파일로 내보내기
    output_filename = "03.통합_건축물대장_표제부.csv"
    df_final_integrated.to_csv(output_filename, index=False, encoding='utf-8-sig')


    print(" 모든 자치구 표제부 데이터 통합 완료")
    print(f" 최종 생성된 마스터 파일명: {output_filename}")
    print(f" 통합된 총 건축물(동) 데이터 수: {total_rows:,}개")



25
  -> [중랑구]03. 표제부_20260521104159.csv
  -> [강북구]03. 표제부_20260521102625.csv
  -> [영등포구]03. 표제부_20260521104057.csv
  -> [성북구]03. 표제부_20260521104018.csv
  -> [은평구]03. 표제부_20260521104119.csv
  -> [용산구]03. 표제부_20260521104106.csv
  -> [강남구]03. 표제부_20260521102538.csv
  -> [구로구]03. 표제부_20260521102813.csv
  -> [서대문구]03. 표제부_20260521103924.csv
  -> [관악구]03. 표제부_20260521102653.csv
  -> [송파구]03. 표제부_20260521104026.csv
  -> [동작구]03. 표제부_20260521103812.csv
  -> [중구]03. 표제부_20260521104151.csv
  -> [강동구]03. 표제부_20260521102549.csv
  -> [도봉구]03. 표제부_20260521103749.csv
  -> [광진구]03. 표제부_20260521102712.csv
  -> [양천구]03. 표제부_20260521104046.csv
  -> [금천구]03. 표제부_20260521102831.csv
  -> [노원구]03. 표제부_20260521102841.csv
  -> [종로구]03. 표제부_20260521104126.csv
  -> [마포구]03. 표제부_20260521103904.csv
  -> [강서구]03. 표제부_20260521102635.csv
  -> [성동구]0

## 기존데이터 csv화

In [14]:
df_23_1 = pd.read_excel('23년 1차.xlsx')
df_23_1.to_csv('23년 1차.csv', index=False, encoding='utf-8-sig')

In [15]:
df_23_2 = pd.read_excel('23년 2차.xlsx')
df_23_2.to_csv('23년 2차.csv', index=False, encoding='utf-8-sig')

In [16]:
df_24_1 = pd.read_excel('24년 1차.xlsx')
df_24_1.to_csv('24년 1차.csv', index=False, encoding='utf-8-sig')

In [17]:
df_24_2 = pd.read_excel('24년 2차.xlsx')
df_24_2.to_csv('24년 2차.csv', index=False, encoding='utf-8-sig')

In [18]:
df_25_1 = pd.read_excel('25년 1차.xlsx')
df_25_1.to_csv('25년 1차.csv', index=False, encoding='utf-8-sig')

In [19]:
df_25_2 = pd.read_excel('25년 2차.xlsx')
df_25_2.to_csv('25년 2차.csv', index=False, encoding='utf-8-sig')

## 법정동코드 정리

In [20]:
import re
import requests
import io

def get_legal_dong_df():

    df = pd.read_csv("법정동코드 전체자료.txt", sep='\t', encoding='cp949')

    return df[df['폐지여부'] == '존재']

def parse_sh_address(address_str):
    #정규표현식 주소 구분
    if pd.isna(address_str):
        return None, None, None, None

    # 괄호 안의 지번 주소 영역 추출 (예: "성내동 521-3")
    # 괄호가 없거나 매칭이 안될 경우를 대비해 도로명 뒤의 지번 패턴 방어코드
    match = re.search(r'\(([^)]+)\)', address_str)

    if match:
        jibun_area = match.group(1)
        # 괄호 내부 텍스트
        # 주택명 등이 섞여 있을 수 있으므로 콤마(,) 기준 앞부분 혹은 동+숫자 패턴 추출
        # ex. 펠리체65 / 천호동 191-13 / 암사동 478-6, 7
        parts = [p.strip() for p in jibun_area.split(',')]

        target_sub = ""
        for p in parts:
            if any(x in p for x in ['동 ', '가 ', '로 ']) and re.search(r'\d', p):
                target_sub = p
                break
        if not target_sub:
            target_sub = parts[-1] if len(parts) > 1 else parts[0]
    else:
        # 괄호가 없는 주소 처리 (도로명만 있거나 일반 지번 주소인 경우)
        target_sub = address_str

    # 법정동명과 번지 분리 정규식
    # (읍/면/동/가/로) + 한 칸 공백 + (숫자-숫자 혹은 숫자)
    pattern = r'([가-힣\d\s]+(?:동|가|로|길))(?:\s+)(\d+)?(?:-(\d+))?'
    res = re.search(pattern, target_sub)

    # 자치구 추출 (시군구 코드용)
    sigungu_match = re.search(r'서울특별시\s+([가-힣]+구)', address_str)
    sigungu = sigungu_match.group(1) if sigungu_match else None

    if res:
        dong = res.group(1).strip()
        # "천호동 191-13"에서 "천호동" 뒤에 주택명이 붙은 경우 처리
        dong = dong.split()[-1]
        # 마지막 단어만 동 이름으로 인정

        bun = res.group(2) if res.group(2) else "0000"
        ji = res.group(3) if res.group(3) else "0000"

        # 4자리 포맷팅 (건축물대장 API 규격 맞춤: 123 -> 0123)
        bun = f"{int(bun):04d}" if bun != "0000" else "0000"
        ji = f"{int(ji):04d}" if ji != "0000" else "0000"

        return sigungu, dong, bun, ji

    return sigungu, None, "0000", "0000"

def process_sh_file(file_path, legal_dong_df):

    # SH 주택목록 상단 헤더 무시하고 데이터 영역만 읽기 (연번이 있는 행부터)
    # 업로드해주신 파일 구조상 상단 2~3줄이 제목이므로 skiprows 사용
    df = pd.read_csv(file_path, skiprows=2)

    # 컬럼명 정제 (공백 제거)
    df.columns = [col.strip() for col in df.columns]

    # 주소 컬럼명 찾기 ('소재지 주소' 혹은 '주소')
    addr_col = [c for c in df.columns if '주소' in c][0]

    print(f"[{file_path}] 처리중인 데이터 개수: {len(df)}")

    # 전처리 적용
    parsed_data = df[addr_col].apply(parse_sh_address)

    df['자치구_추출'] = [x[0] for x in parsed_data]
    df['법정동_추출'] = [x[1] for x in parsed_data]
    df['본번'] = [x[2] for x in parsed_data]
    df['부번'] = [x[3] for x in parsed_data]

    # 법정동 코드를 매칭하기 위한 키 생성
    # ex. 서울특별시 강동구 성내동
    df['검색법정동명'] = "서울특별시 " + df['자치구_추출'] + " " + df['법정동_추출']

    # 법정동 코드 데이터와 병합
    # 주민등록인구용/건축물대장용 10자리 코드 확보
    result_df = pd.merge(df, legal_dong_df[['법정동명', '법정동코드']],
                         left_on='검색법정동명', right_on='법정동명', how='left')

    # 국토부 API용 시군구코드(5자리), 법정동코드(5자리) 쪼개기
    result_df['법정동코드'] = result_df['법정동코드'].astype(str).str.replace('.0', '', regex=False)
    result_df['시군구코드'] = result_df['법정동코드'].str[:5]
    result_df['법정동삼 세부코드'] = result_df['법정동코드'].str[5:]

    # 정제 후 불필요한 임시 컬럼 삭제
    result_df.drop(columns=['검색법정동명', '법정동명'], inplace=True, errors='ignore')

    return result_df


if __name__ == "__main__":
    # 기준 법정동 코드 가져오기
    legal_df = get_legal_dong_df()

    #연도별 파일
    sample_file = "23년 1차.csv"

    try:
        final_df = process_sh_file(sample_file, legal_df)

        # 결과물 확인을 위한 샘플 출력
        print("\n 전처리 및 코드 매칭 완료 샘플 (상위 5개):")
        print(final_df[['자치구', '소재지 주소', '법정동코드', '시군구코드', '법정동삼 세부코드', '본번', '부번']].head())

        # 새로운 파일로 저장
        final_df.to_csv("23년_1차_정제된_SH주택목록_API준비.csv", index=False, encoding='utf-8-sig')
        print("\n '23년_1차_정제된_SH주택목록_API준비.csv' 저장 완료")

    except Exception as e:
        print(f"\n[오류 발생] 파일 양식을 다시 확인해 주세요: {e}")

[23년 1차.csv] 처리중인 데이터 개수: 529

 전처리 및 코드 매칭 완료 샘플 (상위 5개):
   자치구                                  소재지 주소       법정동코드  시군구코드 법정동삼 세부코드  \
0  NaN                                     NaN         nan    nan             
1  NaN                                     NaN         nan    nan             
2  NaN                                     NaN         nan    nan             
3  강동구  서울특별시 강동구 구천면로53길 36-19 (암사동 478-6, 7)  1174010700  11740     10700   
4  강동구  서울특별시 강동구 구천면로53길 36-19 (암사동 478-6, 7)  1174010700  11740     10700   

     본번    부번  
0  None  None  
1  None  None  
2  None  None  
3  0478  0006  
4  0478  0006  

 '23년_1차_정제된_SH주택목록_API준비.csv' 저장 완료


In [21]:
import re
import requests
import io

def get_legal_dong_df():

    df = pd.read_csv("법정동코드 전체자료.txt", sep='\t', encoding='cp949')

    return df[df['폐지여부'] == '존재']

def parse_sh_address(address_str):
    #정규표현식 주소 구분
    if pd.isna(address_str):
        return None, None, None, None

    # 괄호 안의 지번 주소 영역 추출 (예: "성내동 521-3")
    # 괄호가 없거나 매칭이 안될 경우를 대비해 도로명 뒤의 지번 패턴 방어코드
    match = re.search(r'\(([^)]+)\)', address_str)

    if match:
        jibun_area = match.group(1)
        # 괄호 내부 텍스트
        # 주택명 등이 섞여 있을 수 있으므로 콤마(,) 기준 앞부분 혹은 동+숫자 패턴 추출
        # ex. 펠리체65 / 천호동 191-13 / 암사동 478-6, 7
        parts = [p.strip() for p in jibun_area.split(',')]

        target_sub = ""
        for p in parts:
            if any(x in p for x in ['동 ', '가 ', '로 ']) and re.search(r'\d', p):
                target_sub = p
                break
        if not target_sub:
            target_sub = parts[-1] if len(parts) > 1 else parts[0]
    else:
        # 괄호가 없는 주소 처리 (도로명만 있거나 일반 지번 주소인 경우)
        target_sub = address_str

    # 법정동명과 번지 분리 정규식
    # (읍/면/동/가/로) + 한 칸 공백 + (숫자-숫자 혹은 숫자)
    pattern = r'([가-힣\d\s]+(?:동|가|로|길))(?:\s+)(\d+)?(?:-(\d+))?'
    res = re.search(pattern, target_sub)

    # 자치구 추출 (시군구 코드용)
    sigungu_match = re.search(r'서울특별시\s+([가-힣]+구)', address_str)
    sigungu = sigungu_match.group(1) if sigungu_match else None

    if res:
        dong = res.group(1).strip()
        # "천호동 191-13"에서 "천호동" 뒤에 주택명이 붙은 경우 처리
        dong = dong.split()[-1]
        # 마지막 단어만 동 이름으로 인정

        bun = res.group(2) if res.group(2) else "0000"
        ji = res.group(3) if res.group(3) else "0000"

        # 4자리 포맷팅 (건축물대장 API 규격 맞춤: 123 -> 0123)
        bun = f"{int(bun):04d}" if bun != "0000" else "0000"
        ji = f"{int(ji):04d}" if ji != "0000" else "0000"

        return sigungu, dong, bun, ji

    return sigungu, None, "0000", "0000"

def process_sh_file(file_path, legal_dong_df):

    # SH 주택목록 상단 헤더 무시하고 데이터 영역만 읽기 (연번이 있는 행부터)
    # 업로드해주신 파일 구조상 상단 2~3줄이 제목이므로 skiprows 사용
    df = pd.read_csv(file_path, skiprows=2)

    # 컬럼명 정제 (공백 제거)
    df.columns = [col.strip() for col in df.columns]

    # 주소 컬럼명 찾기 ('소재지 주소' 혹은 '주소')
    addr_col = [c for c in df.columns if '주소' in c][0]

    print(f"[{file_path}] 처리중인 데이터 개수: {len(df)}")

    # 전처리 적용
    parsed_data = df[addr_col].apply(parse_sh_address)

    df['자치구_추출'] = [x[0] for x in parsed_data]
    df['법정동_추출'] = [x[1] for x in parsed_data]
    df['본번'] = [x[2] for x in parsed_data]
    df['부번'] = [x[3] for x in parsed_data]

    # 법정동 코드를 매칭하기 위한 키 생성
    # ex. 서울특별시 강동구 성내동
    df['검색법정동명'] = "서울특별시 " + df['자치구_추출'] + " " + df['법정동_추출']

    # 법정동 코드 데이터와 병합
    # 주민등록인구용/건축물대장용 10자리 코드 확보
    result_df = pd.merge(df, legal_dong_df[['법정동명', '법정동코드']],
                         left_on='검색법정동명', right_on='법정동명', how='left')

    # 국토부 API용 시군구코드(5자리), 법정동코드(5자리) 쪼개기
    result_df['법정동코드'] = result_df['법정동코드'].astype(str).str.replace('.0', '', regex=False)
    result_df['시군구코드'] = result_df['법정동코드'].str[:5]
    result_df['법정동삼 세부코드'] = result_df['법정동코드'].str[5:]

    # 정제 후 불필요한 임시 컬럼 삭제
    result_df.drop(columns=['검색법정동명', '법정동명'], inplace=True, errors='ignore')

    return result_df


if __name__ == "__main__":
    # 기준 법정동 코드 가져오기
    legal_df = get_legal_dong_df()

    #연도별 파일
    sample_file = "23년 2차.csv"

    try:
        final_df = process_sh_file(sample_file, legal_df)

        # 결과물 확인을 위한 샘플 출력
        print("\n 전처리 및 코드 매칭 완료 샘플 (상위 5개):")
        print(final_df[['자치구', '소재지 주소', '법정동코드', '시군구코드', '법정동삼 세부코드', '본번', '부번']].head())

        # 새로운 파일로 저장
        final_df.to_csv("23년_2차_정제된_SH주택목록_API준비.csv", index=False, encoding='utf-8-sig')
        print("\n '23년_2차_정제된_SH주택목록_API준비.csv' 저장 완료")

    except Exception as e:
        print(f"\n[오류 발생] 파일 양식을 다시 확인해 주세요: {e}")

[23년 2차.csv] 처리중인 데이터 개수: 581

 전처리 및 코드 매칭 완료 샘플 (상위 5개):
   자치구                                     소재지 주소       법정동코드  시군구코드  \
0  NaN                                        NaN         nan    nan   
1  NaN                                        NaN         nan    nan   
2  NaN                                        NaN         nan    nan   
3  강동구  서울특별시 강동구 진황도로27가길 29 (펠리체65, 천호동 191-13)  1174010900  11740   
4  강동구  서울특별시 강동구 진황도로27가길 29 (펠리체65, 천호동 191-13)  1174010900  11740   

  법정동삼 세부코드    본번    부번  
0            None  None  
1            None  None  
2            None  None  
3     10900  0191  0013  
4     10900  0191  0013  

 '23년_2차_정제된_SH주택목록_API준비.csv' 저장 완료


In [22]:
import re
import requests
import io

def get_legal_dong_df():

    df = pd.read_csv("법정동코드 전체자료.txt", sep='\t', encoding='cp949')

    return df[df['폐지여부'] == '존재']

def parse_sh_address(address_str):
    #정규표현식 주소 구분
    if pd.isna(address_str):
        return None, None, None, None

    # 괄호 안의 지번 주소 영역 추출 (예: "성내동 521-3")
    # 괄호가 없거나 매칭이 안될 경우를 대비해 도로명 뒤의 지번 패턴 방어코드
    match = re.search(r'\(([^)]+)\)', address_str)

    if match:
        jibun_area = match.group(1)
        # 괄호 내부 텍스트
        # 주택명 등이 섞여 있을 수 있으므로 콤마(,) 기준 앞부분 혹은 동+숫자 패턴 추출
        # ex. 펠리체65 / 천호동 191-13 / 암사동 478-6, 7
        parts = [p.strip() for p in jibun_area.split(',')]

        target_sub = ""
        for p in parts:
            if any(x in p for x in ['동 ', '가 ', '로 ']) and re.search(r'\d', p):
                target_sub = p
                break
        if not target_sub:
            target_sub = parts[-1] if len(parts) > 1 else parts[0]
    else:
        # 괄호가 없는 주소 처리 (도로명만 있거나 일반 지번 주소인 경우)
        target_sub = address_str

    # 법정동명과 번지 분리 정규식
    # (읍/면/동/가/로) + 한 칸 공백 + (숫자-숫자 혹은 숫자)
    pattern = r'([가-힣\d\s]+(?:동|가|로|길))(?:\s+)(\d+)?(?:-(\d+))?'
    res = re.search(pattern, target_sub)

    # 자치구 추출 (시군구 코드용)
    sigungu_match = re.search(r'서울특별시\s+([가-힣]+구)', address_str)
    sigungu = sigungu_match.group(1) if sigungu_match else None

    if res:
        dong = res.group(1).strip()
        # "천호동 191-13"에서 "천호동" 뒤에 주택명이 붙은 경우 처리
        dong = dong.split()[-1]
        # 마지막 단어만 동 이름으로 인정

        bun = res.group(2) if res.group(2) else "0000"
        ji = res.group(3) if res.group(3) else "0000"

        # 4자리 포맷팅 (건축물대장 API 규격 맞춤: 123 -> 0123)
        bun = f"{int(bun):04d}" if bun != "0000" else "0000"
        ji = f"{int(ji):04d}" if ji != "0000" else "0000"

        return sigungu, dong, bun, ji

    return sigungu, None, "0000", "0000"

def process_sh_file(file_path, legal_dong_df):

    # SH 주택목록 상단 헤더 무시하고 데이터 영역만 읽기 (연번이 있는 행부터)
    # 업로드해주신 파일 구조상 상단 2~3줄이 제목이므로 skiprows 사용
    df = pd.read_csv(file_path, skiprows=2)

    # 컬럼명 정제 (공백 제거)
    df.columns = [col.strip() for col in df.columns]

    # 주소 컬럼명 찾기 ('소재지 주소' 혹은 '주소')
    addr_col = [c for c in df.columns if '주소' in c][0]

    print(f"[{file_path}] 처리중인 데이터 개수: {len(df)}")

    # 전처리 적용
    parsed_data = df[addr_col].apply(parse_sh_address)

    df['자치구_추출'] = [x[0] for x in parsed_data]
    df['법정동_추출'] = [x[1] for x in parsed_data]
    df['본번'] = [x[2] for x in parsed_data]
    df['부번'] = [x[3] for x in parsed_data]

    # 법정동 코드를 매칭하기 위한 키 생성
    # ex. 서울특별시 강동구 성내동
    df['검색법정동명'] = "서울특별시 " + df['자치구_추출'] + " " + df['법정동_추출']

    # 법정동 코드 데이터와 병합
    # 주민등록인구용/건축물대장용 10자리 코드 확보
    result_df = pd.merge(df, legal_dong_df[['법정동명', '법정동코드']],
                         left_on='검색법정동명', right_on='법정동명', how='left')

    # 국토부 API용 시군구코드(5자리), 법정동코드(5자리) 쪼개기
    result_df['법정동코드'] = result_df['법정동코드'].astype(str).str.replace('.0', '', regex=False)
    result_df['시군구코드'] = result_df['법정동코드'].str[:5]
    result_df['법정동삼 세부코드'] = result_df['법정동코드'].str[5:]

    # 정제 후 불필요한 임시 컬럼 삭제
    result_df.drop(columns=['검색법정동명', '법정동명'], inplace=True, errors='ignore')

    return result_df


if __name__ == "__main__":
    # 기준 법정동 코드 가져오기
    legal_df = get_legal_dong_df()

    #연도별 파일
    sample_file = "24년 1차.csv"

    try:
        final_df = process_sh_file(sample_file, legal_df)

        # 결과물 확인을 위한 샘플 출력
        print("\n 전처리 및 코드 매칭 완료 샘플 (상위 5개):")
        print(final_df[['자치구', '소재지 주소', '법정동코드', '시군구코드', '법정동삼 세부코드', '본번', '부번']].head())

        # 새로운 파일로 저장
        final_df.to_csv("24년_1차_정제된_SH주택목록_API준비.csv", index=False, encoding='utf-8-sig')
        print("\n '24년_1차_정제된_SH주택목록_API준비.csv' 저장 완료")

    except Exception as e:
        print(f"\n[오류 발생] 파일 양식을 다시 확인해 주세요: {e}")

[24년 1차.csv] 처리중인 데이터 개수: 673

 전처리 및 코드 매칭 완료 샘플 (상위 5개):
   자치구                                             소재지 주소       법정동코드  시군구코드  \
0  NaN                                                NaN         nan    nan   
1  NaN                                                NaN         nan    nan   
2  NaN                                                NaN         nan    nan   
3  강동구  서울특별시 강동구 구천면로35길 24-9 (천호동 299-13 외 1필지, 와이디하...  1174010900  11740   
4  강동구  서울특별시 강동구 구천면로35길 24-9 (천호동 299-13 외 1필지, 와이디하...  1174010900  11740   

  법정동삼 세부코드    본번    부번  
0            None  None  
1            None  None  
2            None  None  
3     10900  0299  0013  
4     10900  0299  0013  

 '24년_1차_정제된_SH주택목록_API준비.csv' 저장 완료


In [23]:
import re
import requests
import io

def get_legal_dong_df():

    df = pd.read_csv("법정동코드 전체자료.txt", sep='\t', encoding='cp949')

    return df[df['폐지여부'] == '존재']

def parse_sh_address(address_str):
    #정규표현식 주소 구분
    if pd.isna(address_str):
        return None, None, None, None

    # 괄호 안의 지번 주소 영역 추출 (예: "성내동 521-3")
    # 괄호가 없거나 매칭이 안될 경우를 대비해 도로명 뒤의 지번 패턴 방어코드
    match = re.search(r'\(([^)]+)\)', address_str)

    if match:
        jibun_area = match.group(1)
        # 괄호 내부 텍스트
        # 주택명 등이 섞여 있을 수 있으므로 콤마(,) 기준 앞부분 혹은 동+숫자 패턴 추출
        # ex. 펠리체65 / 천호동 191-13 / 암사동 478-6, 7
        parts = [p.strip() for p in jibun_area.split(',')]

        target_sub = ""
        for p in parts:
            if any(x in p for x in ['동 ', '가 ', '로 ']) and re.search(r'\d', p):
                target_sub = p
                break
        if not target_sub:
            target_sub = parts[-1] if len(parts) > 1 else parts[0]
    else:
        # 괄호가 없는 주소 처리 (도로명만 있거나 일반 지번 주소인 경우)
        target_sub = address_str

    # 법정동명과 번지 분리 정규식
    # (읍/면/동/가/로) + 한 칸 공백 + (숫자-숫자 혹은 숫자)
    pattern = r'([가-힣\d\s]+(?:동|가|로|길))(?:\s+)(\d+)?(?:-(\d+))?'
    res = re.search(pattern, target_sub)

    # 자치구 추출 (시군구 코드용)
    sigungu_match = re.search(r'서울특별시\s+([가-힣]+구)', address_str)
    sigungu = sigungu_match.group(1) if sigungu_match else None

    if res:
        dong = res.group(1).strip()
        # "천호동 191-13"에서 "천호동" 뒤에 주택명이 붙은 경우 처리
        dong = dong.split()[-1]
        # 마지막 단어만 동 이름으로 인정

        bun = res.group(2) if res.group(2) else "0000"
        ji = res.group(3) if res.group(3) else "0000"

        # 4자리 포맷팅 (건축물대장 API 규격 맞춤: 123 -> 0123)
        bun = f"{int(bun):04d}" if bun != "0000" else "0000"
        ji = f"{int(ji):04d}" if ji != "0000" else "0000"

        return sigungu, dong, bun, ji

    return sigungu, None, "0000", "0000"

def process_sh_file(file_path, legal_dong_df):

    # SH 주택목록 상단 헤더 무시하고 데이터 영역만 읽기 (연번이 있는 행부터)
    # 업로드해주신 파일 구조상 상단 2~3줄이 제목이므로 skiprows 사용
    df = pd.read_csv(file_path, skiprows=2)

    # 컬럼명 정제 (공백 제거)
    df.columns = [col.strip() for col in df.columns]

    # 주소 컬럼명 찾기 ('소재지 주소' 혹은 '주소')
    addr_col = [c for c in df.columns if '주소' in c][0]

    print(f"[{file_path}] 처리중인 데이터 개수: {len(df)}")

    # 전처리 적용
    parsed_data = df[addr_col].apply(parse_sh_address)

    df['자치구_추출'] = [x[0] for x in parsed_data]
    df['법정동_추출'] = [x[1] for x in parsed_data]
    df['본번'] = [x[2] for x in parsed_data]
    df['부번'] = [x[3] for x in parsed_data]

    # 법정동 코드를 매칭하기 위한 키 생성
    # ex. 서울특별시 강동구 성내동
    df['검색법정동명'] = "서울특별시 " + df['자치구_추출'] + " " + df['법정동_추출']

    # 법정동 코드 데이터와 병합
    # 주민등록인구용/건축물대장용 10자리 코드 확보
    result_df = pd.merge(df, legal_dong_df[['법정동명', '법정동코드']],
                         left_on='검색법정동명', right_on='법정동명', how='left')

    # 국토부 API용 시군구코드(5자리), 법정동코드(5자리) 쪼개기
    result_df['법정동코드'] = result_df['법정동코드'].astype(str).str.replace('.0', '', regex=False)
    result_df['시군구코드'] = result_df['법정동코드'].str[:5]
    result_df['법정동삼 세부코드'] = result_df['법정동코드'].str[5:]

    # 정제 후 불필요한 임시 컬럼 삭제
    result_df.drop(columns=['검색법정동명', '법정동명'], inplace=True, errors='ignore')

    return result_df


if __name__ == "__main__":
    # 기준 법정동 코드 가져오기
    legal_df = get_legal_dong_df()

    #연도별 파일
    sample_file = "24년 2차.csv"

    try:
        final_df = process_sh_file(sample_file, legal_df)

        # 결과물 확인을 위한 샘플 출력
        print("\n 전처리 및 코드 매칭 완료 샘플 (상위 5개):")
        print(final_df[['자치구', '소재지 주소', '법정동코드', '시군구코드', '법정동삼 세부코드', '본번', '부번']].head())

        # 새로운 파일로 저장
        final_df.to_csv("24년_2차_정제된_SH주택목록_API준비.csv", index=False, encoding='utf-8-sig')
        print("\n '24년_2차_정제된_SH주택목록_API준비.csv' 저장 완료")

    except Exception as e:
        print(f"\n[오류 발생] 파일 양식을 다시 확인해 주세요: {e}")

[24년 2차.csv] 처리중인 데이터 개수: 555

 전처리 및 코드 매칭 완료 샘플 (상위 5개):
   자치구                                     소재지 주소       법정동코드  시군구코드  \
0  NaN                                        NaN         nan    nan   
1  NaN                                        NaN         nan    nan   
2  NaN                                        NaN         nan    nan   
3  강동구  서울특별시 강동구 양재대로143길 31(아성플러스, 명일동 312-143)  1174010100  11740   
4  강동구  서울특별시 강동구 양재대로143길 31(아성플러스, 명일동 312-143)  1174010100  11740   

  법정동삼 세부코드    본번    부번  
0            None  None  
1            None  None  
2            None  None  
3     10100  0312  0143  
4     10100  0312  0143  

 '24년_2차_정제된_SH주택목록_API준비.csv' 저장 완료


In [24]:
import re
import requests
import io

def get_legal_dong_df():

    df = pd.read_csv("법정동코드 전체자료.txt", sep='\t', encoding='cp949')

    return df[df['폐지여부'] == '존재']

def parse_sh_address(address_str):
    #정규표현식 주소 구분
    if pd.isna(address_str):
        return None, None, None, None

    # 괄호 안의 지번 주소 영역 추출 (예: "성내동 521-3")
    # 괄호가 없거나 매칭이 안될 경우를 대비해 도로명 뒤의 지번 패턴 방어코드
    match = re.search(r'\(([^)]+)\)', address_str)

    if match:
        jibun_area = match.group(1)
        # 괄호 내부 텍스트
        # 주택명 등이 섞여 있을 수 있으므로 콤마(,) 기준 앞부분 혹은 동+숫자 패턴 추출
        # ex. 펠리체65 / 천호동 191-13 / 암사동 478-6, 7
        parts = [p.strip() for p in jibun_area.split(',')]

        target_sub = ""
        for p in parts:
            if any(x in p for x in ['동 ', '가 ', '로 ']) and re.search(r'\d', p):
                target_sub = p
                break
        if not target_sub:
            target_sub = parts[-1] if len(parts) > 1 else parts[0]
    else:
        # 괄호가 없는 주소 처리 (도로명만 있거나 일반 지번 주소인 경우)
        target_sub = address_str

    # 법정동명과 번지 분리 정규식
    # (읍/면/동/가/로) + 한 칸 공백 + (숫자-숫자 혹은 숫자)
    pattern = r'([가-힣\d\s]+(?:동|가|로|길))(?:\s+)(\d+)?(?:-(\d+))?'
    res = re.search(pattern, target_sub)

    # 자치구 추출 (시군구 코드용)
    sigungu_match = re.search(r'서울특별시\s+([가-힣]+구)', address_str)
    sigungu = sigungu_match.group(1) if sigungu_match else None

    if res:
        dong = res.group(1).strip()
        # "천호동 191-13"에서 "천호동" 뒤에 주택명이 붙은 경우 처리
        dong = dong.split()[-1]
        # 마지막 단어만 동 이름으로 인정

        bun = res.group(2) if res.group(2) else "0000"
        ji = res.group(3) if res.group(3) else "0000"

        # 4자리 포맷팅 (건축물대장 API 규격 맞춤: 123 -> 0123)
        bun = f"{int(bun):04d}" if bun != "0000" else "0000"
        ji = f"{int(ji):04d}" if ji != "0000" else "0000"

        return sigungu, dong, bun, ji

    return sigungu, None, "0000", "0000"

def process_sh_file(file_path, legal_dong_df):

    # SH 주택목록 상단 헤더 무시하고 데이터 영역만 읽기 (연번이 있는 행부터)
    # 업로드해주신 파일 구조상 상단 2~3줄이 제목이므로 skiprows 사용
    df = pd.read_csv(file_path, skiprows=2)

    # 컬럼명 정제 (공백 제거)
    df.columns = [col.strip() for col in df.columns]

    # 주소 컬럼명 찾기 ('소재지 주소' 혹은 '주소')
    addr_col = [c for c in df.columns if '주소' in c][0]

    print(f"[{file_path}] 처리중인 데이터 개수: {len(df)}")

    # 전처리 적용
    parsed_data = df[addr_col].apply(parse_sh_address)

    df['자치구_추출'] = [x[0] for x in parsed_data]
    df['법정동_추출'] = [x[1] for x in parsed_data]
    df['본번'] = [x[2] for x in parsed_data]
    df['부번'] = [x[3] for x in parsed_data]

    # 법정동 코드를 매칭하기 위한 키 생성
    # ex. 서울특별시 강동구 성내동
    df['검색법정동명'] = "서울특별시 " + df['자치구_추출'] + " " + df['법정동_추출']

    # 법정동 코드 데이터와 병합
    # 주민등록인구용/건축물대장용 10자리 코드 확보
    result_df = pd.merge(df, legal_dong_df[['법정동명', '법정동코드']],
                         left_on='검색법정동명', right_on='법정동명', how='left')

    # 국토부 API용 시군구코드(5자리), 법정동코드(5자리) 쪼개기
    result_df['법정동코드'] = result_df['법정동코드'].astype(str).str.replace('.0', '', regex=False)
    result_df['시군구코드'] = result_df['법정동코드'].str[:5]
    result_df['법정동삼 세부코드'] = result_df['법정동코드'].str[5:]

    # 정제 후 불필요한 임시 컬럼 삭제
    result_df.drop(columns=['검색법정동명', '법정동명'], inplace=True, errors='ignore')

    return result_df


if __name__ == "__main__":
    # 기준 법정동 코드 가져오기
    legal_df = get_legal_dong_df()

    #연도별 파일
    sample_file = "25년 1차.csv"

    try:
        final_df = process_sh_file(sample_file, legal_df)

        # 결과물 확인을 위한 샘플 출력
        print("\n 전처리 및 코드 매칭 완료 샘플 (상위 5개):")
        print(final_df[['자치구', '소재지 주소', '법정동코드', '시군구코드', '법정동삼 세부코드', '본번', '부번']].head())

        # 새로운 파일로 저장
        final_df.to_csv("25년_1차_정제된_SH주택목록_API준비.csv", index=False, encoding='utf-8-sig')
        print("\n '25년_1차_정제된_SH주택목록_API준비.csv' 저장 완료")

    except Exception as e:
        print(f"\n[오류 발생] 파일 양식을 다시 확인해 주세요: {e}")

[25년 1차.csv] 처리중인 데이터 개수: 754

 전처리 및 코드 매칭 완료 샘플 (상위 5개):
   자치구                           소재지 주소       법정동코드  시군구코드 법정동삼 세부코드    본번  \
0  NaN                              NaN         nan    nan            None   
1  NaN                              NaN         nan    nan            None   
2  NaN                              NaN         nan    nan            None   
3  강동구  서울특별시 강동구 풍성로42길 12 (성내동 521-3)  1174010800  11740     10800  0521   
4  강동구  서울특별시 강동구 풍성로42길 12 (성내동 521-3)  1174010800  11740     10800  0521   

     부번  
0  None  
1  None  
2  None  
3  0003  
4  0003  

 '25년_1차_정제된_SH주택목록_API준비.csv' 저장 완료


In [25]:
import re
import requests
import io

def get_legal_dong_df():

    df = pd.read_csv("법정동코드 전체자료.txt", sep='\t', encoding='cp949')

    return df[df['폐지여부'] == '존재']

def parse_sh_address(address_str):
    #정규표현식 주소 구분
    if pd.isna(address_str):
        return None, None, None, None

    # 괄호 안의 지번 주소 영역 추출 (예: "성내동 521-3")
    # 괄호가 없거나 매칭이 안될 경우를 대비해 도로명 뒤의 지번 패턴 방어코드
    match = re.search(r'\(([^)]+)\)', address_str)

    if match:
        jibun_area = match.group(1)
        # 괄호 내부 텍스트
        # 주택명 등이 섞여 있을 수 있으므로 콤마(,) 기준 앞부분 혹은 동+숫자 패턴 추출
        # ex. 펠리체65 / 천호동 191-13 / 암사동 478-6, 7
        parts = [p.strip() for p in jibun_area.split(',')]

        target_sub = ""
        for p in parts:
            if any(x in p for x in ['동 ', '가 ', '로 ']) and re.search(r'\d', p):
                target_sub = p
                break
        if not target_sub:
            target_sub = parts[-1] if len(parts) > 1 else parts[0]
    else:
        # 괄호가 없는 주소 처리 (도로명만 있거나 일반 지번 주소인 경우)
        target_sub = address_str

    # 법정동명과 번지 분리 정규식
    # (읍/면/동/가/로) + 한 칸 공백 + (숫자-숫자 혹은 숫자)
    pattern = r'([가-힣\d\s]+(?:동|가|로|길))(?:\s+)(\d+)?(?:-(\d+))?'
    res = re.search(pattern, target_sub)

    # 자치구 추출 (시군구 코드용)
    sigungu_match = re.search(r'서울특별시\s+([가-힣]+구)', address_str)
    sigungu = sigungu_match.group(1) if sigungu_match else None

    if res:
        dong = res.group(1).strip()
        # "천호동 191-13"에서 "천호동" 뒤에 주택명이 붙은 경우 처리
        dong = dong.split()[-1]
        # 마지막 단어만 동 이름으로 인정

        bun = res.group(2) if res.group(2) else "0000"
        ji = res.group(3) if res.group(3) else "0000"

        # 4자리 포맷팅 (건축물대장 API 규격 맞춤: 123 -> 0123)
        bun = f"{int(bun):04d}" if bun != "0000" else "0000"
        ji = f"{int(ji):04d}" if ji != "0000" else "0000"

        return sigungu, dong, bun, ji

    return sigungu, None, "0000", "0000"

def process_sh_file(file_path, legal_dong_df):

    # SH 주택목록 상단 헤더 무시하고 데이터 영역만 읽기 (연번이 있는 행부터)
    # 업로드해주신 파일 구조상 상단 2~3줄이 제목이므로 skiprows 사용
    df = pd.read_csv(file_path, skiprows=2)

    # 컬럼명 정제 (공백 제거)
    df.columns = [col.strip() for col in df.columns]

    # 주소 컬럼명 찾기 ('소재지 주소' 혹은 '주소')
    addr_col = [c for c in df.columns if '주소' in c][0]

    print(f"[{file_path}] 처리중인 데이터 개수: {len(df)}")

    # 전처리 적용
    parsed_data = df[addr_col].apply(parse_sh_address)

    df['자치구_추출'] = [x[0] for x in parsed_data]
    df['법정동_추출'] = [x[1] for x in parsed_data]
    df['본번'] = [x[2] for x in parsed_data]
    df['부번'] = [x[3] for x in parsed_data]

    # 법정동 코드를 매칭하기 위한 키 생성
    # ex. 서울특별시 강동구 성내동
    df['검색법정동명'] = "서울특별시 " + df['자치구_추출'] + " " + df['법정동_추출']

    # 법정동 코드 데이터와 병합
    # 주민등록인구용/건축물대장용 10자리 코드 확보
    result_df = pd.merge(df, legal_dong_df[['법정동명', '법정동코드']],
                         left_on='검색법정동명', right_on='법정동명', how='left')

    # 국토부 API용 시군구코드(5자리), 법정동코드(5자리) 쪼개기
    result_df['법정동코드'] = result_df['법정동코드'].astype(str).str.replace('.0', '', regex=False)
    result_df['시군구코드'] = result_df['법정동코드'].str[:5]
    result_df['법정동삼 세부코드'] = result_df['법정동코드'].str[5:]

    # 정제 후 불필요한 임시 컬럼 삭제
    result_df.drop(columns=['검색법정동명', '법정동명'], inplace=True, errors='ignore')

    return result_df


if __name__ == "__main__":
    # 기준 법정동 코드 가져오기
    legal_df = get_legal_dong_df()

    #연도별 파일
    sample_file = "25년 2차.csv"

    try:
        final_df = process_sh_file(sample_file, legal_df)

        # 결과물 확인을 위한 샘플 출력
        print("\n 전처리 및 코드 매칭 완료 샘플 (상위 5개):")
        print(final_df[['자치구', '소재지 주소', '법정동코드', '시군구코드', '법정동삼 세부코드', '본번', '부번']].head())

        # 새로운 파일로 저장
        final_df.to_csv("25년_2차_정제된_SH주택목록_API준비.csv", index=False, encoding='utf-8-sig')
        print("\n '25년_2차_정제된_SH주택목록_API준비.csv' 저장 완료")

    except Exception as e:
        print(f"\n[오류 발생] 파일 양식을 다시 확인해 주세요: {e}")

[25년 2차.csv] 처리중인 데이터 개수: 719

 전처리 및 코드 매칭 완료 샘플 (상위 5개):
   자치구                            소재지 주소       법정동코드  시군구코드 법정동삼 세부코드    본번  \
0  NaN                               NaN         nan    nan            None   
1  NaN                               NaN         nan    nan            None   
2  NaN                               NaN         nan    nan            None   
3  강동구  서울특별시 강동구 상암로35길 10   (천호동 38-8)  1174010900  11740     10900  0038   
4  강동구  서울특별시 강동구 상암로35길 10   (천호동 38-8)  1174010900  11740     10900  0038   

     부번  
0  None  
1  None  
2  None  
3  0008  
4  0008  

 '25년_2차_정제된_SH주택목록_API준비.csv' 저장 완료


## SH주택목록 통합

In [26]:
import pandas as pd
import glob
import os

# 통합할 SH 파일 탐색 규칙 설정
file_pattern = "*SH주택목록*.csv"
file_list = glob.glob(file_pattern)


print(f"발견된 SH 주택목록 파일 개수: {len(file_list)}개")

for f in file_list:
    print(f"  -> {f}")


if len(file_list) == 0:
    print("폴더에 'SH주택목록' 문구가 포함된 CSV 파일이 없음")
else:
    all_sh_data = []

    print("\n파일별 가변 헤더 자동 감지 및 데이터 추출 시작...")
    for file in file_list:
        print(f"   [처리 중] {file} ...")


        # 연번 찾기 헤더 없이 상위 10줄
        df_preview = pd.read_csv(file, header=None, nrows=10)

        # '연번'이나 '연번'과 유사한 단어가 들어있는 행의 인덱스
        header_idx = None
        for idx, row in df_preview.iterrows():
            if row.astype(str).str.contains('연번').any():
                header_idx = idx
                break

        # 연번 없을 경우 기본값 3
        if header_idx is None:
            print(f" '연번' 행을 찾지 못해 기본 규칙을 적용합니다.")
            header_idx = 3
        else:
            print(f" '연번' 행 자동 감지 완료! (상단 {header_idx}행 건너뜀)")

        # 찾아낸 위치를 기준으로 데이터를 재 로딩
        df = pd.read_csv(file, header=header_idx)

        # 양쪽 공백 때문에 컬럼명이 안 잡히는 현상 방지
        df.columns = df.columns.str.strip()

        #'연번이 유효한 숫자 데이터인 행만 필터링
        df = df[pd.to_numeric(df['연번'], errors='coerce').notna()].copy()

        # 결측치 정제 및 컬럼 포맷 통일
        df['주택명'] = df['주택명'].fillna('명칭없음').astype(str).str.strip()
        df['소재지 주소'] = df['소재지 주소'].fillna('').astype(str).str.strip()
        if '호' in df.columns:
            df['호'] = df['호'].fillna('').astype(str).str.split('.').str[0].str.strip()
        else:
            df['호'] = ""

        # 출처 기입
        df['데이터출처_파일명'] = os.path.basename(file)

        all_sh_data.append(df)
        print(f" (유효 데이터: {len(df):,}개)")

    # 데이터 위아래로 통합
    print("\n전 차수 데이터 통합 진행 중...")
    df_integrated_sh = pd.concat(all_sh_data, ignore_index=True)

    # 불필요한 시스템 자동 생성 컬럼 제거
    unnamed_cols = [col for col in df_integrated_sh.columns if 'Unnamed' in col]
    df_integrated_sh.drop(columns=unnamed_cols, inplace=True, errors='ignore')

    before_total = len(df_integrated_sh)


    print("\n데이터 정제 및 중복 제거(Deduplication) 가동...")

    # 완전 동일 행 제거
    df_integrated_sh = df_integrated_sh.drop_duplicates(keep='first')
    after_step1 = len(df_integrated_sh)

    # 주소, 주택명, 호수, 전용면적이 모두 겹치는 중복 매물 제거
    dup_keys = ['소재지 주소', '주택명', '호', '전용면적(㎡)']

    # 존재하는 컬럼들만 기준으로 중복 제거 안전장치
    dup_keys_sh = [k for k in dup_keys if k in df_integrated_sh.columns]
    df_integrated_sh = df_integrated_sh.drop_duplicates(subset=dup_keys_sh, keep='first')
    after_step2 = len(df_integrated_sh)

    # 연번 재정렬
    df_integrated_sh = df_integrated_sh.reset_index(drop=True)
    df_integrated_sh['연번'] = df_integrated_sh.index + 1

    # 최종 결과물 CSV 저장
    output_name = "[master]통합_SH주택목록_마스터_최종.csv"
    df_integrated_sh.to_csv(output_name, index=False, encoding='utf-8-sig')

    removed_total = before_total - after_step2


    print(" 중복 정제 및 전 차수 SH 데이터 통합 완료")
    print(f" 최종 파일명: {output_name}")
    print(f" 최초 누적 데이터 합계: {before_total:,}개")
    print(f" 제거된 총 중복 데이터: {removed_total:,}개")
    print(f" 최종 정제된 유니크 매물 수: {after_step2:,}개")

발견된 SH 주택목록 파일 개수: 6개
  -> 23년_1차_정제된_SH주택목록_API준비.csv
  -> 25년_1차_정제된_SH주택목록_API준비.csv
  -> 24년_1차_정제된_SH주택목록_API준비.csv
  -> 25년_2차_정제된_SH주택목록_API준비.csv
  -> 24년_2차_정제된_SH주택목록_API준비.csv
  -> 23년_2차_정제된_SH주택목록_API준비.csv

파일별 가변 헤더 자동 감지 및 데이터 추출 시작...
   [처리 중] 23년_1차_정제된_SH주택목록_API준비.csv ...
 '연번' 행 자동 감지 완료! (상단 0행 건너뜀)
 (유효 데이터: 526개)
   [처리 중] 25년_1차_정제된_SH주택목록_API준비.csv ...
 '연번' 행 자동 감지 완료! (상단 0행 건너뜀)
 (유효 데이터: 751개)
   [처리 중] 24년_1차_정제된_SH주택목록_API준비.csv ...
 '연번' 행 자동 감지 완료! (상단 0행 건너뜀)
 (유효 데이터: 670개)
   [처리 중] 25년_2차_정제된_SH주택목록_API준비.csv ...
 '연번' 행 자동 감지 완료! (상단 0행 건너뜀)
 (유효 데이터: 716개)
   [처리 중] 24년_2차_정제된_SH주택목록_API준비.csv ...
 '연번' 행 자동 감지 완료! (상단 0행 건너뜀)
 (유효 데이터: 552개)
   [처리 중] 23년_2차_정제된_SH주택목록_API준비.csv ...
 '연번' 행 자동 감지 완료! (상단 0행 건너뜀)
 (유효 데이터: 578개)

전 차수 데이터 통합 진행 중...

데이터 정제 및 중복 제거(Deduplication) 가동...
 중복 정제 및 전 차수 SH 데이터 통합 완료
 최종 파일명: [master]통합_SH주택목록_마스터_최종.csv
 최초 누적 데이터 합계: 3,793개
 제거된 총 중복 데이터: 0개
 최종 정제된 유니크 매물 수: 3,793개


## 파일 매칭

In [27]:
import pandas as pd
import numpy as np
import re

# 파일 경로 설정
SH_MASTER_FILE = "[master]통합_SH주택목록_마스터_최종.csv"
LEDGER_FILE = "03.통합_건축물대장_표제부.csv"

print("데이터 로드 및 1차 정제")

# 데이터 불러오기
df_sh = pd.read_csv(SH_MASTER_FILE)
df_ledger = pd.read_csv(LEDGER_FILE, low_memory=False)

# 지번 패딩 및 문자열 정제 전처리 함수들
def clean_to_string(val):
    if pd.isna(val): return ""
    return str(val).split('.')[0].strip()

def pad_bun_ji(val):
    st = clean_to_string(val)
    if st.isdigit(): return f"{int(st):04d}"
    return "0000"

# SH 데이터 주소 키 표준화
df_sh['시군구_key'] = df_sh['시군구코드'].apply(clean_to_string)
df_sh['법정동_key'] = df_sh['법정동삼 세부코드'].apply(clean_to_string)
df_sh['본번_key'] = df_sh['본번'].apply(pad_bun_ji)
df_sh['부번_key'] = df_sh['부번'].apply(pad_bun_ji)

# 건축물대장 데이터 주소 키 표준화
df_ledger['시군구_key'] = df_ledger['시군구코드'].apply(clean_to_string)
df_ledger['법정동_key'] = df_ledger['법정동코드'].apply(clean_to_string)
df_ledger['본번_key'] = df_ledger['번'].apply(pad_bun_ji)
df_ledger['부번_key'] = df_ledger['지'].apply(pad_bun_ji)

# 대장에서 필요한 핵심 노후도 지표 컬럼들만 도려내기 (사용승인일, 구조, 높이, 층수 등)
# 허가번호년이나 착공일 등 필요한 시계열 데이터가 있다면 리스트에 추가 가능
ledger_cols = ['시군구_key', '법정동_key', '본번_key', '부번_key', '도로명대지위치', '건물명', '사용승인일', '구조코드명', '지상층수']
df_ledger_sub = df_ledger[ledger_cols].copy()

# 데이터 공백 제거
df_ledger_sub['도로명대지위치'] = df_ledger_sub['도로명대지위치'].fillna('').astype(str).str.replace(" ", "")
df_ledger_sub['건물명'] = df_ledger_sub['건물명'].fillna('').astype(str).str.replace(" ", "")
df_sh['소재지 주소_clean'] = df_sh['소재지 주소'].fillna('').astype(str).str.replace(" ", "")
df_sh['주택명_clean'] = df_sh['주택명'].fillna('').astype(str).str.replace(" ", "")

print(f"   SH 매물 수: {len(df_sh):,}개 / 통합 대장 건축물 수: {len(df_ledger_sub):,}개")

# 결과 담을 빈 컬럼 생성
df_sh['사용승인일'] = np.nan
df_sh['구조코드명'] = np.nan
df_sh['지상층수'] = np.nan
df_sh['매칭단계'] = '미매칭'


print("\n 코드 매칭 ")

# 고유 지번코드 매칭 (시군구+법정동+본번+부번)
# 대장에서 가장 확실한 유니크 지번 1개씩만 남기기
df_ledger_idx1 = df_ledger_sub.drop_duplicates(subset=['시군구_key', '법정동_key', '본번_key', '부번_key'], keep='first')

df_m1 = pd.merge(df_sh, df_ledger_idx1, on=['시군구_key', '법정동_key', '본번_key', '부번_key'], how='left', suffixes=('', '_m1'))
idx_m1 = df_m1['사용승인일_m1'].notna()

df_sh.loc[idx_m1, '사용승인일'] = df_m1.loc[idx_m1, '사용승인일_m1']
df_sh.loc[idx_m1, '구조코드명'] = df_m1.loc[idx_m1, '구조코드명_m1']
df_sh.loc[idx_m1, '지상층수'] = df_m1.loc[idx_m1, '지상층수_m1']
df_sh.loc[idx_m1, '매칭단계'] = '1단계(지번코드)'
print(f"1단계(지번코드): {idx_m1.sum():,}개")

# 도로명 주소 기반 텍스트 매칭
still_missing = df_sh['사용승인일'].isna()
if still_missing.any():
    df_ledger_idx2 = df_ledger_sub[df_ledger_sub['도로명대지위치'] != ''].drop_duplicates(subset=['도로명대지위치'], keep='first')

    df_m2 = pd.merge(df_sh[still_missing], df_ledger_idx2, left_on='소재지 주소_clean', right_on='도로명대지위치', how='left', suffixes=('', '_m2'))
    idx_m2 = df_m2['사용승인일_m2'].notna()

    # 원본 인덱스 역추적하여 매핑
    real_indices = df_sh[still_missing].index[idx_m2]
    df_sh.loc[real_indices, '사용승인일'] = df_m2.loc[idx_m2, '사용승인일_m2'].values
    df_sh.loc[real_indices, '구조코드명'] = df_m2.loc[idx_m2, '구조코드명_m2'].values
    df_sh.loc[real_indices, '지상층수'] = df_m2.loc[idx_m2, '지상층수_m2'].values
    df_sh.loc[real_indices, '매칭단계'] = '2단계(도로명주소)'
    print(f"2단계(도로명주소): {idx_m2.sum():,}개")

# 자치구 + 텍스트 주택명 기반 매칭
still_missing = df_sh['사용승인일'].isna()
if still_missing.any():
    # 주택명이 명칭없음이 아니고 빈칸이 아닌 대장 데이터 매핑용 베이스 구축
    df_ledger_idx3 = df_ledger_sub[(df_ledger_sub['건물명'] != '') & (df_ledger_sub['건물명'] != '명칭없음')]
    df_ledger_idx3 = df_ledger_idx3.drop_duplicates(subset=['시군구_key', '건물명'], keep='first')

    df_m3 = pd.merge(df_sh[still_missing], df_ledger_idx3, on=['시군구_key'], how='left', suffixes=('', '_m3'))

    # 주택명 텍스트 매칭 일치 조건 탐색
    idx_m3 = (df_m3['주택명_clean'] != '명칭없음') & (df_m3['주택명_clean'] != '') & (df_m3['주택명_clean'] == df_m3['건물명'])

    # 겹치는 것 중 중복 제거 후 원본 프레임에 주입
    df_m3_success = df_m3[idx_m3].drop_duplicates(subset=['연번'], keep='first').set_index('연번')

    for idx, row in df_m3_success.iterrows():
        # 연번 기준으로 원본 행 추적
        orig_idx = df_sh[df_sh['연번'] == idx].index
        if len(orig_idx) > 0:
            df_sh.loc[orig_idx, '사용승인일'] = row['사용승인일_m3']
            df_sh.loc[orig_idx, '구조코드명'] = row['구조코드명_m3']
            df_sh.loc[orig_idx, '지상층수'] = row['지상층수_m3']
            df_sh.loc[orig_idx, '매칭단계'] = '3단계(텍스트주택명)'

    print(f"3단계(텍스트주택명): {len(df_m3_success):,}개")


# 불필요하게 임시 생성한 전처리 매칭용 보조 키 컬럼 삭제
drop_temp_cols = ['시군구_key', '법정동_key', '본번_key', '부번_key', '소재지 주소_clean', '주택명_clean']
df_sh.drop(columns=drop_temp_cols, inplace=True, errors='ignore')

# 최종 통계치 계산
success_count = len(df_sh[df_sh['매칭단계'] != '미매칭'])
success_rate = (success_count / len(df_sh)) * 100

output_filename = "[노후도결합]SH주택목록_대장매칭완료.csv"
df_sh.to_csv(output_filename, index=False, encoding='utf-8-sig')


print("하이브리드 다중 매칭 정보 결합 완료")
print(f" 최종 통합본 파일명: {output_filename}")
print(f" 분석된 SH 총 유니크 주택 수: {len(df_sh):,}개")
print(f" 최종 통합 매칭 성공률: {success_count:,}개 ({success_rate:.1f}%)")


데이터 로드 및 1차 정제
   SH 매물 수: 3,793개 / 통합 대장 건축물 수: 586,457개

 코드 매칭 


/tmp/ipykernel_1111/1469512777.py:66: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['20221013' '20221013' '20221013' ... '20220707' '20220902' '20220902']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_sh.loc[idx_m1, '사용승인일'] = df_m1.loc[idx_m1, '사용승인일_m1']
/tmp/ipykernel_1111/1469512777.py:67: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['철근콘크리트구조' '철근콘크리트구조' '철근콘크리트구조' ... '철근콘크리트구조' '철근콘크리트구조' '철근콘크리트구조']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_sh.loc[idx_m1, '구조코드명'] = df_m1.loc[idx_m1, '구조코드명_m1']


1단계(지번코드): 3,734개
2단계(도로명주소): 0개
3단계(텍스트주택명): 2개
하이브리드 다중 매칭 정보 결합 완료
 최종 통합본 파일명: [노후도결합]SH주택목록_대장매칭완료.csv
 분석된 SH 총 유니크 주택 수: 3,793개
 최종 통합 매칭 성공률: 3,736개 (98.5%)


## 커트라인데이터 CSV 파일로 추출

In [1]:
import os
import pandas as pd

def excel_to_csv_all_sheets(excel_file_path, output_directory="."):
    """
    엑셀 파일의 모든 시트를 각각 CSV 파일로 변환하여 저장하는 함수

    :param excel_file_path: 변환할 엑셀 파일 경로
    :param output_directory: CSV 파일들을 저장할 폴더 경로 (기본값은 현재 폴더)
    """
    # 1. 파일 존재 여부 확인
    if not os.path.exists(excel_file_path):
        print(f"❌ 파일을 찾을 수 없습니다: {excel_file_path}")
        return

    # 2. 출력 폴더가 없으면 생성
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
        print(f"📁 저장 폴더를 생성했습니다: {output_directory}")

    try:
        # 3. 엑셀 파일의 모든 시트를 딕셔너리 형태로 로드 (sheet_name=None 필수)
        # {시트이름: 데이터프레임} 구조로 가져옵니다.
        excel_data = pd.read_excel(excel_file_path, sheet_name=None)

        print(f"🔍 엑셀 로드 완료! 총 {len(excel_data)}개의 시트를 발견했습니다.\n")

        # 4. 각 시트별로 반복하며 CSV로 저장
        for sheet_name, df in excel_data.items():
            # 파일명에 사용할 수 없는 특수문자 제거 및 공백 정리
            safe_sheet_name = "".join([c for c in sheet_name if c.isalpha() or c.isdigit() or c in " _-"]).strip()

            # 원본 엑셀 파일명에서 확장자를 제외한 이름 추출
            base_name = os.path.splitext(os.path.basename(excel_file_path))[0]

            # 최종 CSV 파일명 생성 (예: 커트라인 데이터 - 2025-1.csv)
            csv_file_name = f"{base_name} - {safe_sheet_name}.csv"
            csv_file_path = os.path.join(output_directory, csv_file_name)

            # 한글 깨짐을 방지하기 위해 'utf-8-sig' 인코딩으로 저장
            df.to_csv(csv_file_path, index=False, encoding='utf-8-sig')
            print(f"✅ 변환 완료: {csv_file_name} ({len(df)}행)")

        print("\n🎉 모든 시트가 성공적으로 CSV 파일로 변환되었습니다!")

    except Exception as e:
        print(f"❌ 오류 발생: {e}")

# ====================================================
# [사용 예시]
# ====================================================
# 변환하고자 하는 엑셀 파일명을 여기에 적어주세요.
target_excel = "커트라인 데이터 (6) (1) (1).xlsx"

# 함수 실행
excel_to_csv_all_sheets(target_excel)

🔍 엑셀 로드 완료! 총 9개의 시트를 발견했습니다.

✅ 변환 완료: 커트라인 데이터 (6) (1) (1) - 2021-1.csv (33행)
✅ 변환 완료: 커트라인 데이터 (6) (1) (1) - 2021-2.csv (84행)
✅ 변환 완료: 커트라인 데이터 (6) (1) (1) - 2022-1.csv (104행)
✅ 변환 완료: 커트라인 데이터 (6) (1) (1) - 2022-2.csv (113행)
✅ 변환 완료: 커트라인 데이터 (6) (1) (1) - 2023-1.csv (89행)
✅ 변환 완료: 커트라인 데이터 (6) (1) (1) - 2023-2.csv (136행)
✅ 변환 완료: 커트라인 데이터 (6) (1) (1) - 2024-1.csv (132행)
✅ 변환 완료: 커트라인 데이터 (6) (1) (1) - 2024-2.csv (125행)
✅ 변환 완료: 커트라인 데이터 (6) (1) (1) - 2025-1.csv (145행)

🎉 모든 시트가 성공적으로 CSV 파일로 변환되었습니다!


In [4]:
import os
import re
import pandas as pd

def clean_address(addr):
    if pd.isna(addr):
        return ""
    addr = str(addr).strip()

    # 1. '서울특별시' 명칭 통일
    if not (addr.startswith("서울특별시") or addr.startswith("서울시") or addr.startswith("서울 ")):
        addr = "서울특별시 " + addr
    else:
        addr = re.sub(r'^서울시?\s+', '서울특별시 ', addr)

    # 2. 괄호 및 대괄호 내용 제거
    addr = re.sub(r'\(.*?\)', '', addr)
    addr = re.sub(r'\[.*?\]', '', addr)

    # 3. 쉼표 뒤 상세주소 제거
    if ',' in addr:
        addr = addr.split(',')[0]

    # 4. 연속된 공백 하나로 축소
    addr = re.sub(r'\s+', ' ', addr).strip()
    return addr

# 1. SH 주택목록 파일 로드
sh_file = '[노후도결합]SH주택목록_대장매창(승인연도).csv'
sh_df = pd.read_csv(sh_file, encoding='utf-8-sig')
sh_addresses = sh_df['소재지 주소'].dropna().unique()

# 2. 가지고 계신 여러 개의 커트라인 파일 리스트 (파일명들을 여기에 적어주세요)
cutline_files = [
    "커트라인 데이터 (6) (1) (1) - 2023-1.csv",
    "커트라인 데이터 (6) (1) (1) - 2023-2.csv",
    "커트라인 데이터 (6) (1) (1) - 2024-1.csv",
    "커트라인 데이터 (6) (1) (1) - 2024-2.csv",
    "커트라인 데이터 (6) (1) (1) - 2025-1.csv",
]

# 3. 파일들 반복문으로 읽어서 하나로 합치기
cutline_list = []
for file in cutline_files:
    if os.path.exists(file):
        # 파일 구조(헤더 위치) 반영하여 로드
        df = pd.read_csv(file, header=1, encoding='utf-8-sig')
        df.columns = df.columns.str.strip()

        if '주소지' in df.columns:
            cutline_list.append(df[['주소지']].dropna())
            print(f"✅ 파일 로드 성공: {file} ({len(df)}개 행)")
    else:
        print(f"❌ 파일을 찾을 수 없음: {file}")

# 통합 데이터 생성
if not cutline_list:
    raise FileNotFoundError("로딩에 성공한 커트라인 데이터가 없습니다.")

combined_cut_df = pd.concat(cutline_list, ignore_index=True)
cut_addresses = combined_cut_df['주소지'].unique()

# 4. 핵심 주소 일치 계산
cleaned_sh_set = set(clean_address(addr) for addr in sh_addresses if clean_address(addr))
cleaned_cut_set = set(clean_address(addr) for addr in cut_addresses if clean_address(addr))
common_core_addresses = cleaned_sh_set.intersection(cleaned_cut_set)

sh_df['정제주소'] = sh_df['소재지 주소'].apply(clean_address)
core_match_sh_df = sh_df[sh_df['정제주소'].isin(common_core_addresses)]

# 5. 최종 결과 출력
print("\n" + "=" * 50)
print(f"▶ 통합된 커트라인 전체 주소 행 개수: {len(combined_cut_df):,}개")
print(f"▶ SH 주택목록과 핵심주소 일치하는 행 개수: {len(core_match_sh_df):,}개")
print("=" * 50)

✅ 파일 로드 성공: 커트라인 데이터 (6) (1) (1) - 2023-1.csv (88개 행)
✅ 파일 로드 성공: 커트라인 데이터 (6) (1) (1) - 2023-2.csv (135개 행)
✅ 파일 로드 성공: 커트라인 데이터 (6) (1) (1) - 2024-1.csv (131개 행)
✅ 파일 로드 성공: 커트라인 데이터 (6) (1) (1) - 2024-2.csv (124개 행)
✅ 파일 로드 성공: 커트라인 데이터 (6) (1) (1) - 2025-1.csv (144개 행)

▶ 통합된 커트라인 전체 주소 행 개수: 622개
▶ SH 주택목록과 핵심주소 일치하는 행 개수: 3,249개


## 노후도 상관계수 분석

In [34]:
import os
import re
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

In [35]:
def extract_address_key(addr):
    if pd.isna(addr):
        return None
    addr = str(addr).strip()

    # 서울특별시 + 자치구 추출
    gu_match = re.search(r'(서울특별시|서울시|서울)\s+([가-힣]+구)', addr)
    if gu_match:
        gu_part = f"서울특별시 {gu_match.group(2)}"
    else:
        # 주소 처음에 서울이 생략된 경우 자치구만 찾기
        gu_match_direct = re.search(r'([가-힣]+구)', addr)
        if gu_match_direct:
            gu_part = f"서울특별시 {gu_match_direct.group(1)}"
        else:
            gu_part = "서울특별시 미상구"

    # 괄호 () 안의 세부 내용 추출 (지번, 건물명 등)
    # 여러 개의 괄호가 있을 수 있으므로 모두 추출하여 공백 제거 후 결합
    parentheses = re.findall(r'\((.*?)\)', addr)
    if parentheses:
        # 괄호 안의 불필요한 내용 정제
        cleaned_parents = []
        for p in parentheses:
            p_clean = re.sub(r'\s*외\s*\d*필지|\s*외|\s*,\s*|,\d+동|제\d+동', '', p).strip()
            if p_clean and p_clean != '명칭없음':
                cleaned_parents.append(p_clean)
        parentheses_part = "".join(cleaned_parents).replace(" ", "")
    else:
        parentheses_part = ""

    # 만약 괄호 내용이 없다면 도로명/지번 뒷부분 일부를 보조 키로 활용
    if not parentheses_part:
        # 괄호가 없는 경우 주소의 맨 끝 단어 2개를 공백 없이 붙임
        chunks = [c for c in addr.split() if c not in ['서울특별시', '서울시', '서울']]
        parentheses_part = "".join(chunks[-2:]) if len(chunks) >= 2 else "".join(chunks)

    return f"{gu_part}_{parentheses_part}"

## 주택목록 로드 변수 생성

In [36]:
sh_file = '[노후도결합]SH주택목록_대장매창(승인연도).csv'
if not os.path.exists(sh_file):
    sh_file = '[노후도결합]SH주택목록_대장매칭완료.csv'


sh_df = pd.read_csv(sh_file, encoding='utf-8-sig')

# 사용승인일 정제 및 노후도 계산 (2026년 기준)
sh_df['사용승인년도'] = pd.to_numeric(sh_df['사용승인일'].astype(str).str[:4], errors='coerce')
sh_df = sh_df.dropna(subset=['사용승인년도'])
sh_df['Housing_Age'] = 2026 - sh_df['사용승인년도']

# 대조용 매칭 키 생성
sh_df['Matching_Key'] = sh_df['소재지 주소'].apply(extract_address_key)

# 주택(건물) 단위 분석을 위해 매칭키 기준 중복 제거
sh_buildings = sh_df.drop_duplicates(subset=['Matching_Key']).copy()
print(f"SH 주택목록 고유 건물 수: {len(sh_buildings)}개\n")

SH 주택목록 고유 건물 수: 464개



In [37]:
all_files = os.listdir('.')
cutline_files = [
    f for f in all_files
    if ('커트라인' in f or '커트라인' in f)
    and f.endswith('.csv')
    and any(yr in f for yr in ['2023', '2024', '2025'])
    and f != sh_file
]

# 파일명 정렬해서 차수 순서대로 정렬
cutline_files = sorted(cutline_files)

matched_data_list = []

In [39]:
for file in cutline_files:
    # 파일명에서 연도-차수 식별자 추출
    label_match = re.search(r'202\d-\d', file)
    label = label_match.group(0) if label_match else file

    try:
        # 헤더가 고정되어 있지 않으므로 유연하게 탐색
        raw_df = pd.read_csv(file, header=None, encoding='utf-8-sig')

        # 주소지, 순위, 가점 인덱스 찾기
        idx_addr, idx_rank, idx_score = -1, -1, -1
        for row_idx in range(5):
            row_str = raw_df.iloc[row_idx].astype(str).str.strip().tolist()
            for col_idx, cell in enumerate(row_str):
                if '주소지' in cell: idx_addr = col_idx
                if cell in ['순위', '신청순위']: idx_rank = col_idx
                if '가점' in cell: idx_score = col_idx
            if idx_addr != -1 and idx_rank != -1 and idx_score != -1:
                data_start_idx = row_idx + 1
                break

        if idx_addr == -1 or idx_rank == -1 or idx_score == -1:
            print(f"{label} 매칭 실패") # 에러 시 구조 확인
            continue

        # 실제 데이터 추출
        cut_df = raw_df.iloc[data_start_idx:].copy()
        cut_df.columns = raw_df.iloc[data_start_idx-1]

        # 필요한 정보 추출 및 정제
        temp_df = pd.DataFrame()
        temp_df['Raw_Address'] = cut_df.iloc[:, idx_addr].astype(str).str.strip()
        temp_df['Rank'] = cut_df.iloc[:, idx_rank].astype(str).str.strip()
        temp_df['Raw_Score'] = cut_df.iloc[:, idx_score].astype(str).str.replace(r'[^\d]', '', regex=True)
        temp_df['Raw_Score'] = pd.to_numeric(temp_df['Raw_Score'], errors='coerce')
        temp_df['Matching_Key'] = temp_df['Raw_Address'].apply(extract_address_key)

        temp_df = temp_df.dropna(subset=['Raw_Score', 'Matching_Key'])

        # SH 주택목록 건물과 커트라인 주소 매칭
        merged_sub = pd.merge(sh_buildings[['Matching_Key', 'Housing_Age']], temp_df, on='Matching_Key', how='inner')

        # 중복 매칭 제거
        merged_sub = merged_sub.drop_duplicates(subset=['Matching_Key', 'Rank'])

        print(f"{label} 차수 유효 매칭 데이터 개수: {len(merged_sub)}개")

        # 전체 분석용 리스트에 추가
        merged_sub['Source_Period'] = label
        matched_data_list.append(merged_sub)

    except Exception as e:
        print(f"{file} 처리 중 에러 발생: {e}")

2023-1 차수 유효 매칭 데이터 개수: 69개
2023-2 차수 유효 매칭 데이터 개수: 77개
2024-1 차수 유효 매칭 데이터 개수: 86개
2024-2 차수 유효 매칭 데이터 개수: 95개
2025-1 차수 유효 매칭 데이터 개수: 113개


In [41]:
def 가점_보정_및_그룹(row):
    rank_str = str(row['Rank'])
    score = row['Raw_Score']

    if '1순위' in rank_str:
        return pd.Series(['Group_1_2', score + 14, '1순위 집단(+14점)'])
    elif '2순위' in rank_str:
        return pd.Series(['Group_1_2', score + 0, '2순위 집단(+0점)'])
    elif '3순위' in rank_str:
        return pd.Series(['Group_3', score, '3순위 집단(독립분리)'])
    else:
        return pd.Series(['Other', np.nan, '기타'])

final_analysis_df[['Group_Type', 'Adjusted_Score', 'Group_Label']] = final_analysis_df.apply(가점_보정_및_그룹, axis=1)
final_analysis_df = final_analysis_df.dropna(subset=['Adjusted_Score'])

In [42]:
analysis_groups = [
    ('1순위(+14점)', final_analysis_df[final_analysis_df['Group_Label'] == '1순위(+14점)']),
    ('2순위(+0점)', final_analysis_df[final_analysis_df['Group_Label'] == '2순위(+0점)']),
    ('3순위(독립분리)', final_analysis_df[final_analysis_df['Group_Label'] == '3순위(독립분리)']),
    ('1/2순위 통합 보정', final_analysis_df[final_analysis_df['Group_Type'] == 'Group_1_2'])
]

In [50]:
for label, sub_df in analysis_groups:
    n_count = len(sub_df)
    if n_count >= 3:
        # 피어슨 상관계수
        corr = sub_df[['Housing_Age', 'Adjusted_Score']].corr().loc['Housing_Age', 'Adjusted_Score']
        print(f"{label}")
        print(f"   샘플 수 : {n_count}개")
        print(f"   피어슨 상관계수 (r) : {corr:.4f}")

        # 상관관계 해석 매핑
        if abs(corr) < 0.1:
            interpretation = "선형 상관관계가 전혀 없음"
        elif abs(corr) < 0.3:
            interpretation = "미약한 상관관계가 존재함"
        else:
            interpretation = "유의미한 선형 상관관계가 확인됨"

    else:
        print(f" {label} 차")
        print(f"   데이터 수 부족으로 분석 불가 (N = {n_count}개)\n")

1순위 집단(+14점)
   샘플 수 : 189개
   피어슨 상관계수 (r) : 0.0705
2순위 집단(+0점)
   샘플 수 : 239개
   피어슨 상관계수 (r) : -0.0384
3순위 집단(독립분리)
   샘플 수 : 12개
   피어슨 상관계수 (r) : -0.1307
[참고] 1·2순위 통합 보정 집단
   샘플 수 : 428개
   피어슨 상관계수 (r) : 0.0487
